# AlphaDent YOLO26 Segmentation Pipeline (End-to-End)

This notebook contains the complete pipeline for downloading the AlphaDent dataset, training, validating, and performing inference using **YOLO26 segment models**.

### Models Supported:
- `yolo26n-seg.pt` (Nano)
- `yolo26s-seg.pt` (Small)
- `yolo26m-seg.pt` (Medium)
- `yolo26l-seg.pt` (Large)
- `yolo26x-seg.pt` (Extra Large)

Developed for running on **Kaggle**.

## 1. Clone Repository & Setup Working Directory

In [ ]:
# Clone repo if not running inside it already
import os
if not os.path.exists('/kaggle/working/AlphaDentYolov26'):
    print("Cloning AlphaDentYolov26 repository...")
    !git clone https://github.com/ShMazumder/AlphaDentYolov26 /kaggle/working/AlphaDentYolov26/
else:
    print("Repository already exists.")

# Change directory to the repository root
os.chdir('/kaggle/working/AlphaDentYolov26/')
print(f"Current working directory: {os.getcwd()}")

# Install dependencies
print("Installing dependencies...")
!pip install -q -r requirements.txt
!pip install -q matplotlib opencv-python

## 2. Configuration

Configure model selection and training hyperparameters here.

In [ ]:
import torch

# Choose model: 'yolo26n-seg.pt', 'yolo26s-seg.pt', 'yolo26m-seg.pt', 'yolo26l-seg.pt', 'yolo26x-seg.pt'
MODEL_NAME = 'yolo26x-seg.pt'

# Dataset configuration file path
DATA_CONFIG = './AlphaDent/yolo_seg_train.yaml'

# Device configuration: list of all available GPU IDs, or 'cpu'
DEVICE = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else 'cpu'

# Training Settings
BATCH_SIZE = 16      # reduce if CUDA out of memory
EPOCHS = 100         # training epochs
IMAGE_SIZE = 640     # input size (640 or 960)

# Dynamic project naming to prevent overwriting results between different models/runs
MODEL_BASE = MODEL_NAME.replace('-seg.pt', '').replace('.pt', '')
PROJECT_NAME = f'yolo_seg_{MODEL_BASE}_proj_{IMAGE_SIZE}'

# Validation & Inference Settings
CONF_THRESHOLD = 0.001
IOU_THRESHOLD = 0.5

print(f"Configured model: {MODEL_NAME}")
print(f"Dataset config: {DATA_CONFIG}")
print(f"Device: {DEVICE}")
print(f"Project directory: {PROJECT_NAME}")
print(f"Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}, Image size: {IMAGE_SIZE}")

## 3. Dataset Download & Unpacking

Download the dataset from Zenodo and extract it to the correct path.

In [ ]:
import zipfile
import shutil
import os

input_dir = "/kaggle/input/competitions/alpha-dent/AlphaDent"
working_dir = "/kaggle/working/AlphaDentYolov26/AlphaDent"
zip_path = "/kaggle/working/dataset.zip"

# Check if Kaggle input contains the dataset
if os.path.exists(os.path.join(input_dir, 'images')) and os.listdir(os.path.join(input_dir, 'images')):
    # print("Dataset found in Kaggle input. Preparing files...")
    # os.makedirs(working_dir, exist_ok=True)
    # # 1. Symlink the large images folder (read-only is fine)
    # src_images = os.path.join(input_dir, 'images')
    # dst_images = os.path.join(working_dir, 'images')
    # if not os.path.exists(dst_images):
    #     os.symlink(src_images, dst_images)
    #     print("Symlinked images directory.")
    # # 2. Copy the small labels folder (needs to be writeable for YOLO cache writing)
    # src_labels = os.path.join(input_dir, 'labels')
    # dst_labels = os.path.join(working_dir, 'labels')
    # if not os.path.exists(dst_labels):
    #     print("Copying labels directory to allow cache writing...")
    #     shutil.copytree(src_labels, dst_labels)
    #     print("Copied labels directory.")
    # print("Dataset prepared successfully using Kaggle input.")

    print("Dataset found in Kaggle input. Copying dataset...")
    shutil.copytree(input_dir, working_dir, dirs_exist_ok=True)
    print("Dataset copied successfully. Using input dataset.")
else:
    # Fallback to downloading from Zenodo
    images_dir = os.path.join(working_dir, 'images')
    if not os.path.exists(images_dir) or not os.listdir(images_dir):
        print("Downloading AlphaDent dataset from Zenodo...")
        !wget -q --show-progress "https://zenodo.org/records/16582489/files/AlphaDent.zip?download=1" -O {zip_path}
        
        print("Extracting dataset...")
        os.makedirs(working_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(working_dir)
        
        # Remove zip to save disk space
        if os.path.exists(zip_path):
            os.remove(zip_path)
            
        # Check if nested folder exists after extraction and move files up if needed
        nested_dir = os.path.join(working_dir, 'AlphaDent')
        if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
            print("Adjusting dataset directory structure...")
            for item in os.listdir(nested_dir):
                shutil.move(os.path.join(nested_dir, item), working_dir)
            os.rmdir(nested_dir)
            
        print("Dataset prepared successfully in working directory.")
    else:
        print("Dataset already exists in working directory.")

## 4. Train Model

Load the selected YOLO26 model and start training.

In [ ]:
import torch
from ultralytics import YOLO

# Ensure W&B is disabled to run offline smoothly
os.environ['WANDB_DISABLED'] = 'true'

# Load a pretrained YOLO26 segment model
print(f"Loading {MODEL_NAME}...")
model = YOLO(MODEL_NAME)

# Start training
print("Starting training...")
results = model.train(
    data=DATA_CONFIG,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    deterministic=False,
    plots=True,
    device=DEVICE,
)
print("Training completed.")

## 5. Model Validation

Evaluate the trained model's performance on the validation dataset.

In [ ]:
# Find best weights by checking multiple potential directories
possible_paths = [
    f"runs/segment/{PROJECT_NAME}/train/weights/best.pt",
    f"{PROJECT_NAME}/train/weights/best.pt",
    f"runs/{PROJECT_NAME}/train/weights/best.pt",
    f"/kaggle/working/AlphaDentYolov26/runs/segment/{PROJECT_NAME}/train/weights/best.pt"
]

weights_path = MODEL_NAME
for p in possible_paths:
    if os.path.exists(p):
        weights_path = p
        print(f"Found trained weights at: {p}")
        break
else:
    print(f"Trained weights not found. Using pretrained: {weights_path}")

# Load model for validation
model_val = YOLO(weights_path)

# Run validation (with self-healing fallback to CPU if CUDA runs out of memory)
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
print(f"Running validation on {val_device}...")

try:
    metrics = model_val.val(
        data=DATA_CONFIG,
        project=PROJECT_NAME,
        imgsz=IMAGE_SIZE,
        batch=1,
        iou=IOU_THRESHOLD,
        conf=CONF_THRESHOLD,
        half=True,
        save_json=True,
        save_txt=True,
        save_conf=True,
        plots=True,
        device=val_device,
    )
except RuntimeError as e:
    if "out of memory" in str(e).lower() and val_device != "cpu":
        print("⚠️ CUDA Out of Memory during validation. Retrying on CPU...")
        gc.collect()
        torch.cuda.empty_cache()
        metrics = model_val.val(
            data=DATA_CONFIG,
            project=PROJECT_NAME,
            imgsz=IMAGE_SIZE,
            batch=1,
            iou=IOU_THRESHOLD,
            conf=CONF_THRESHOLD,
            save_json=True,
            save_txt=True,
            save_conf=True,
            plots=True,
            device="cpu",
        )
    else:
        raise e

print("Validation metrics:")
print(f"Box mAP50-95: {metrics.box.map:.4f}")
print(f"Box mAP50: {metrics.box.map50:.4f}")
print(f"Mask mAP50-95: {metrics.seg.map:.4f}")
print(f"Mask mAP50: {metrics.seg.map50:.4f}")

## 6. Inference and Visualization

Perform inference on test images and display the predictions.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

# Run inference on validation/test images to see quality
test_images_dir = "./AlphaDent/images/valid/"
output_dir = "./inference_output/"

model_infer = YOLO(weights_path)

print(f"Running inference on images from: {test_images_dir}")
val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
results_infer = model_infer.predict(
    source=test_images_dir,
    project=output_dir,
    name='preds',
    imgsz=IMAGE_SIZE,
    conf=0.25,  # higher threshold for visual clarity
    iou=0.6,
    half=True,
    save=True,
    save_txt=True,
    device=val_device
)

# Plot the first few results
saved_images = glob.glob(f"{output_dir}/preds/*.jpg") + glob.glob(f"{output_dir}/preds/*.png")
if saved_images:
    num_display = min(3, len(saved_images))
    fig, axes = plt.subplots(1, num_display, figsize=(18, 6))
    if num_display == 1:
        axes = [axes]
    for idx, img_path in enumerate(saved_images[:num_display]):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(os.path.basename(img_path))
    plt.tight_layout()
    plt.show()
else:
    print("No output images found to display.")

## 7. Model Explainability (XAI)

To build trust in medical diagnosis applications, we must explain the model's predictions. This section implements **post-hoc Explainable AI (XAI)** methods to visualize which image regions the YOLO26 model pays attention to when detecting tooth pathologies.

### Supported CAM Methods:
1. **Eigen-CAM**: Computes principal components of the activations. Gradient-free and extremely robust.
2. **Grad-CAM**: Visualizes gradients of the class score with respect to intermediate activations.
3. **Grad-CAM++**: An advanced gradient-based method using second-order gradients.
4. **XGrad-CAM**: Scales the gradients by the normalized activations.
5. **HiRes-CAM**: Element-wise multiplies activations with gradients; guarantees faithfulness.
6. **Layer-CAM**: Spatially weights activations by positive gradients (excellent for lower/intermediate layers).
7. **EigenGrad-CAM**: First principal component of (Activations * Gradients).

### Literature References:
- **Reference Paper (Medical YOLO XAI)**: Borah et al., 2024. ["A comprehensive study on Explainable AI using YOLO and post hoc method on medical diagnosis"](https://doi.org/10.1088/1742-6596/2919/1/012045).
- **Grad-CAM**: Selvaraju et al., 2017. ["Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization"](https://arxiv.org/abs/1610.02391).
- **Grad-CAM++**: Chattopadhay et al., 2018. ["Grad-CAM++: Generalized Gradient-based Visual Explanations for Deep Convolutional Networks"](https://arxiv.org/abs/1710.11063).
- **XGrad-CAM**: Fu et al., 2020. ["Axiom-based Grad-CAM: Towards Accurate Visualization and Explanation of CNNs"](https://arxiv.org/abs/2008.02312).
- **HiRes-CAM**: Draelos & Carin, 2020. ["Use HiResCAM instead of Grad-CAM for faithful explanations of convolutional neural networks"](https://arxiv.org/abs/2011.08891).
- **Layer-CAM**: Jiang et al., 2021. ["LayerCAM: Exploring Hierarchical Class Activation Maps for Localization"](https://arxiv.org/abs/2104.14818).
- **Eigen-CAM / EigenGrad-CAM**: Muhammad & Yeasin, 2020. ["Eigen-CAM: Class Activation Map using Principal Components"](https://arxiv.org/abs/2008.00299).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

class YOLOExplainer:
    """
    Custom Class Activation Mapping (CAM) generator for Ultralytics YOLO models.
    Supports Eigen-CAM, Grad-CAM, Grad-CAM++, XGrad-CAM, HiRes-CAM, Layer-CAM, and EigenGrad-CAM.
    """
    def __init__(self, model, target_layer, method='eigencam'):
        self.model = model
        self.target_layer = target_layer
        self.method = method.lower()
        self.activations = None
        self.gradients = None
        
        # Register hooks
        self.forward_hook = target_layer.register_forward_hook(self._forward_hook_fn)
        if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
            self.backward_hook = target_layer.register_full_backward_hook(self._backward_hook_fn)
            
    def _forward_hook_fn(self, module, input, output):
        if isinstance(output, (list, tuple)):
            self.activations = output[0].clone()
        else:
            self.activations = output.clone()
            
    def _backward_hook_fn(self, module, grad_input, grad_output):
        if isinstance(grad_output, (list, tuple)):
            self.gradients = grad_output[0].clone()
        else:
            self.gradients = grad_output.clone()
            
    def remove_hooks(self):
        self.forward_hook.remove()
        if hasattr(self, 'backward_hook'):
            self.backward_hook.remove()
            
    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        
        if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
            input_tensor.requires_grad = True
            
        outputs = self.model(input_tensor)
        
        if isinstance(outputs, (list, tuple)):
            preds = outputs[0]
        else:
            preds = outputs
            
        if self.method == 'eigencam':
            act = self.activations.detach().cpu().numpy()[0]
            C, H, W = act.shape
            matrix = act.reshape(C, H * W).T
            matrix = matrix - np.mean(matrix, axis=0)
            U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
            projection = U[:, 0].reshape(H, W)
            heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
            return heatmap
            
        elif self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
            if class_idx is None:
                class_scores = preds[0, 4:13, :].sum(dim=-1)
                class_idx = torch.argmax(class_scores).item()
                
            score = preds[0, 4 + class_idx, :].sum()
            score.backward(retain_graph=True)
            
            act = self.activations.detach().cpu().numpy()[0]
            grad = self.gradients.detach().cpu().numpy()[0]
            
            if self.method == 'gradcam':
                weights = np.mean(grad, axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap
                
            elif self.method == 'gradcam++':
                grads_power_2 = grad ** 2
                grads_power_3 = grad ** 3
                sum_act = np.sum(act, axis=(1, 2))
                alpha = grads_power_2 / (2 * grads_power_2 + sum_act[:, None, None] * grads_power_3 + 1e-8)
                weights = np.sum(alpha * np.maximum(grad, 0), axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'xgradcam':
                sum_act = np.sum(act, axis=(1, 2))
                weights = np.sum(grad * act, axis=(1, 2)) / (sum_act + 1e-8)
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'hirescam':
                cam = np.sum(act * grad, axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'layercam':
                cam = np.sum(act * np.maximum(grad, 0), axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'eigengradcam':
                act_grad = act * np.maximum(grad, 0)
                C, H, W = act_grad.shape
                matrix = act_grad.reshape(C, H * W).T
                matrix = matrix - np.mean(matrix, axis=0)
                U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
                projection = U[:, 0].reshape(H, W)
                heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
                return heatmap

def show_heatmap(img_path, heatmap, title='Attention Map'):
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].axis('off')
    axes[0].set_title('Original Image')
    
    axes[1].imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    axes[1].axis('off')
    axes[1].set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
from ultralytics import YOLO

# 1. Load model with trained weights
model_explain = YOLO(weights_path)

# 2. List model layers to choose from
print("Model Layers:")
for idx, layer in enumerate(model_explain.model.model):
    print(f"Layer {idx}: {type(layer).__name__}")

# 3. Select a target layer (e.g. index 22, the last C3k2 block in YOLO26 neck/backbone)
target_layer_idx = 22
target_layer = model_explain.model.model[target_layer_idx]

# 4. Prepare input image
test_images = glob.glob("./AlphaDent/images/valid/*.jpg") + glob.glob("./AlphaDent/images/valid/*.png")
if test_images:
    img_path = test_images[0]
    print(f"Using image for XAI: {img_path}")
    
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    img_norm = img_resized.astype(np.float32) / 255.0
    input_tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0)
    input_tensor = input_tensor.to(val_device)
    
    # 5. Generate and show heatmaps for different methods
    methods = ['EigenCAM', 'GradCAM', 'GradCAM++', 'XGradCAM', 'HiResCAM', 'LayerCAM', 'EigenGradCAM']
    for method in methods:
        print(f"Generating {method}...")
        explainer = YOLOExplainer(model_explain, target_layer, method=method)
        heatmap = explainer.generate(input_tensor)
        explainer.remove_hooks()
        
        # Show it
        show_heatmap(img_path, heatmap, title=f'{method} Attention Map')
else:
    print("No validation images found to explain.")